In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../../data/raw/diabetes+130-us+hospitals+for+years+1999-2008")
REPORT_DIR = Path("../../reports/tables")
PROCESSED_DIR = Path("../../data/processed")

df = pd.read_csv(RAW_DIR / "diabetic_data.csv")

print(df.shape)
df.head()

In [ ]:
mapping_path = RAW_DIR / "IDs_mapping.csv"

with open(mapping_path, "r", encoding="utf-8") as f:
    mapping_text = f.read()

print(mapping_text[:10000])

In [ ]:
work = df.copy()

work = work.replace("?", np.nan)

In [ ]:
print("Original dataframe:")
print(df.isna().sum().sum())

print("After converting ?:")
print(work.isna().sum().sum())

In [ ]:
missing = pd.DataFrame({
    "missing_n": work.isna().sum(),
    "missing_pct": work.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

display(missing.head(20))

In [ ]:
missing.to_csv(
    REPORT_DIR / "uci_missingness_summary.csv"
)

In [ ]:
rows = []

for col in work.columns:
    rows.append({
        "variable": col,
        "pandas_dtype": str(work[col].dtype),
        "unique_values": work[col].nunique(dropna=True),
        "missing_n": work[col].isna().sum(),
        "missing_pct": round(work[col].isna().mean() * 100, 2),
        "example_values": ", ".join(
            map(str, work[col].dropna().unique()[:5])
        )
    })

dictionary = pd.DataFrame(rows)

dictionary

In [ ]:
missing.to_csv(
    REPORT_DIR / "uci_missingness_summary.csv"
)

In [ ]:
rows = []

for col in work.columns:
    rows.append({
        "variable": col,
        "pandas_dtype": str(work[col].dtype),
        "unique_values": work[col].nunique(dropna=True),
        "missing_n": work[col].isna().sum(),
        "missing_pct": round(work[col].isna().mean() * 100, 2),
        "example_values": ", ".join(
            map(str, work[col].dropna().unique()[:5])
        )
    })

dictionary = pd.DataFrame(rows)

dictionary

In [ ]:
dictionary.to_csv(
    REPORT_DIR / "uci_data_dictionary_draft.csv",
    index=False
)

In [ ]:
work["readmission_30d"] = (
    work["readmitted"] == "<30"
).astype(int)

In [ ]:
target_summary = (
    work["readmission_30d"]
    .value_counts()
    .rename_axis("readmission_30d")
    .reset_index(name="n")
)

target_summary["percentage"] = (
    target_summary["n"] / len(work) * 100
)

target_summary

In [ ]:
target_summary.to_csv(
    REPORT_DIR / "readmission_30d_distribution.csv",
    index=False
)

In [ ]:
patient_encounters = (
    work.groupby("patient_nbr")
    .size()
    .rename("encounter_count")
)

patient_encounters.describe()

In [ ]:
print(
    "Unique patients:",
    work["patient_nbr"].nunique()
)

print(
    "Patients with >1 encounter:",
    (patient_encounters > 1).sum()
)

print(
    "Percentage with >1 encounter:",
    round(
        (patient_encounters > 1).mean() * 100,
        2
    )
)

print(
    "Maximum encounters:",
    patient_encounters.max()
)

In [ ]:
repeat_patients = patient_encounters[
    patient_encounters > 1
].sort_values(ascending=False)

repeat_patients.head(20)

In [ ]:
example_patient = repeat_patients.index[0]

work[
    work["patient_nbr"] == example_patient
][[
    "encounter_id",
    "patient_nbr",
    "age",
    "time_in_hospital",
    "number_inpatient",
    "readmitted",
    "readmission_30d"
]]

In [ ]:
work["discharge_disposition_id"].value_counts().sort_index()

In [ ]:
discharge_analysis = (
    work.groupby("discharge_disposition_id")
    .agg(
        encounters=("encounter_id", "count"),
        readmissions=("readmission_30d", "sum"),
        readmission_rate=("readmission_30d", "mean")
    )
)

discharge_analysis["readmission_rate"] *= 100

discharge_analysis.sort_values(
    "encounters",
    ascending=False
)

In [ ]:
missing[
    missing["missing_pct"] > 0
]

In [ ]:
from pathlib import Path
print(Path.cwd())

In [ ]:
!git status

In [ ]:
!git add .
!git commit -m "Discharge analysis"
!git push origin main

In [ ]:
missing_groups = pd.cut(
    missing["missing_pct"],
    bins=[-0.01, 5, 20, 50, 80, 100],
    labels=[
        "0–5%",
        "5–20%",
        "20–50%",
        "50–80%",
        "80–100%"
    ]
)

missing.assign(
    missing_group=missing_groups
).sort_values(
    "missing_pct",
    ascending=False
)

In [ ]:
identifier_columns = [
    "encounter_id",
    "patient_nbr"
]

In [ ]:
variable_roles = {
    "identifier": [
        "encounter_id",
        "patient_nbr"
    ],

    "demographic": [
        "race",
        "gender",
        "age",
        "weight"
    ],

    "admission": [
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
        "time_in_hospital"
    ],

    "utilization": [
        "number_outpatient",
        "number_emergency",
        "number_inpatient"
    ],

    "clinical_activity": [
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_diagnoses"
    ],

    "diagnosis": [
        "diag_1",
        "diag_2",
        "diag_3"
    ],

    "laboratory": [
        "max_glu_serum",
        "A1Cresult"
    ],

    "target": [
        "readmitted",
        "readmission_30d"
    ]
}

In [ ]:

# Get all column names from the dataset
all_columns = set(df.columns)

# Collect all columns listed in variable_roles
classified_columns = {
    column
    for columns in variable_roles.values()
    for column in columns
}

# Identify columns that exist in the dataset but are not classified
unclassified_columns = sorted(all_columns - classified_columns)

# Identify names in the dictionary that do not exist in the dataset
unknown_columns = sorted(classified_columns - all_columns)

print("Total dataset columns:", len(all_columns))
print("Classified columns:", len(classified_columns))
print("Unclassified columns:", len(unclassified_columns))

print("\nUnclassified columns:")
print(unclassified_columns)

print("\nNames not found in the dataset:")
print(unknown_columns)


In [ ]:

variable_roles = {
    "identifier": [
        "encounter_id",
        "patient_nbr"
    ],

    "demographic": [
        "race",
        "gender",
        "age",
        "weight"
    ],

    "admission": [
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
        "time_in_hospital"
    ],

    "utilization": [
        "number_outpatient",
        "number_emergency",
        "number_inpatient"
    ],

    "clinical_activity": [
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_diagnoses"
    ],

    "diagnosis": [
        "diag_1",
        "diag_2",
        "diag_3"
    ],

    "laboratory": [
        "max_glu_serum",
        "A1Cresult"
    ],

    "medication_management": [
        "change",
        "diabetesMed"
    ],

    "medications": [
        "acarbose",
        "acetohexamide",
        "chlorpropamide",
        "citoglipton",
        "examide",
        "glimepiride",
        "glimepiride-pioglitazone",
        "glipizide",
        "glipizide-metformin",
        "glyburide",
        "glyburide-metformin",
        "insulin",
        "metformin",
        "metformin-pioglitazone",
        "metformin-rosiglitazone",
        "miglitol",
        "nateglinide",
        "pioglitazone",
        "repaglinide",
        "rosiglitazone",
        "tolazamide",
        "tolbutamide",
        "troglitazone"
    ],

    "administrative": [
        "payer_code",
        "medical_specialty"
    ],

    "target": [
        "readmitted"
    ]
}



In [ ]:

all_columns = set(df.columns)

classified_columns = {
    column
    for columns in variable_roles.values()
    for column in columns
}

unclassified_columns = sorted(all_columns - classified_columns)
unknown_columns = sorted(classified_columns - all_columns)

print("Total dataset columns:", len(all_columns))
print("Classified columns:", len(classified_columns))
print("Unclassified columns:", len(unclassified_columns))
print("\nUnclassified columns:", unclassified_columns)
print("Names not found in dataset:", unknown_columns)


In [ ]:

import numpy as np

work = df.copy()

# Convert the dataset's question-mark placeholders to missing values
work = work.replace("?", np.nan)

# Create a binary outcome while preserving the original readmitted column
work["readmission_30d"] = (
    work["readmitted"] == "<30"
).astype(int)

print(work["readmission_30d"].value_counts().sort_index())
print(work["readmission_30d"].value_counts(normalize=True).mul(100).round(2))


### Outcome Definition: 30-Day Readmission

The primary outcome for the initial TRIP research prototype is a binary indicator derived from the original `readmitted` variable in the UCI Diabetes 130-US Hospitals dataset.

- `1`: The encounter was followed by a recorded readmission within 30 days (`readmitted == "<30"`).
- `0`: The encounter was followed by a readmission after 30 days (`">30"`) or had no recorded readmission (`"NO"`).

In the initial dataset, 11,357 of 101,766 encounters (11.16%) were classified as positive, while 90,409 encounters (88.84%) were classified as negative.

The negative category combines late readmissions and no recorded readmission. The outcome therefore measures early readmission against this combined comparison group.

These figures are preliminary descriptive statistics. Cohort eligibility, repeated encounters, missing data, and prediction timing must be examined before model development.

In [ ]:

# 1. Check the original outcome categories
print("Original readmission categories:")
print(work["readmitted"].value_counts(dropna=False))

# 2. Confirm the derived target distribution
print("\nDerived target distribution:")
print(
    work["readmission_30d"]
    .value_counts()
    .sort_index()
)

# 3. Examine the number of encounters and unique patients
print("\nDataset structure:")
print("Total encounters:", len(work))
print("Unique patients:", work["patient_nbr"].nunique())
print(
    "Patients with multiple encounters:",
    (work["patient_nbr"].value_counts() > 1).sum()
)

# 4. Check for missing patient identifiers
print(
    "Missing patient identifiers:",
    work["patient_nbr"].isna().sum()
)


In [ ]:

total_encounters = len(work)
unique_patients = work["patient_nbr"].nunique()

print("Encounters per unique patient:",
      round(total_encounters / unique_patients, 2))

print("Encounters beyond one per patient:",
      total_encounters - unique_patients)


In [ ]:

print("Discharge disposition codes:")
print(
    work["discharge_disposition_id"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMissing values in key variables:")
key_variables = [
    "patient_nbr",
    "readmitted",
    "discharge_disposition_id",
    "admission_type_id",
    "admission_source_id",
    "time_in_hospital"
]

print(work[key_variables].isna().sum())


In [ ]:
print(mapping_text)

In [ ]:

print(mapping_text)


In [ ]:

position = mapping_text.find("discharge_disposition_id")

print("Section position:", position)
print(mapping_text[position:position + 5000])


In [ ]:

!git status



In [ ]:
from pathlib import Path

current = Path.cwd()
repo_root = next(
    (p for p in current.parents if (p / ".git").exists()),
    None
)

print("Repository root:", repo_root)

In [ ]:
notebook = repo_root / "notebooks" / "python" / "02_cohort_and_data_dictionary.ipynb"

print("Notebook exists:", notebook.exists())
print("Notebook path:", notebook)

In [ ]:
!git add 02_cohort_and_data_dictionary.ipynb

In [ ]:
!git commit -m "analysis: update discharge mappings and cohort planning"

In [ ]:
!git push

In [ ]:
# Descriptions from the IDs_mapping.csv file
discharge_descriptions = {
    1: "Discharged to home",
    2: "Transferred to another short-term hospital",
    3: "Transferred to skilled nursing facility",
    4: "Transferred to intermediate care facility",
    5: "Transferred to another inpatient care institution",
    6: "Discharged home with home health service",
    7: "Left against medical advice",
    8: "Discharged home under home IV provider",
    9: "Admitted as inpatient to this hospital",
    10: "Neonatal aftercare at another hospital",
    11: "Expired",
    12: "Still patient or expected to return for outpatient services",
    13: "Hospice at home",
    14: "Hospice in medical facility",
    15: "Transferred to swing bed",
    16: "Referred or transferred for outpatient services",
    17: "Referred or transferred to this institution for outpatient services",
    18: "NULL",
    19: "Expired at home",
    20: "Expired in a medical facility",
    21: "Expired, place unknown",
    22: "Transferred to rehabilitation facility",
    23: "Transferred to long-term care hospital",
    24: "Transferred to Medicaid-certified nursing facility",
    25: "Not mapped",
    26: "Unknown or invalid",
    27: "Transferred to federal healthcare facility",
    28: "Transferred to psychiatric hospital",
    29: "Transferred to critical access hospital",
    30: "Transferred to another healthcare institution"
}

# Count encounters for each discharge code
discharge_summary = (
    work["discharge_disposition_id"]
    .value_counts(dropna=False)
    .rename_axis("discharge_disposition_id")
    .reset_index(name="encounter_count")
)

# Attach the descriptions
discharge_summary["description"] = (
    discharge_summary["discharge_disposition_id"]
    .map(discharge_descriptions)
)

# Calculate percentages
discharge_summary["percentage"] = (
    discharge_summary["encounter_count"] / len(work) * 100
).round(2)

# Display the summary in numerical order
discharge_summary = discharge_summary.sort_values(
    "discharge_disposition_id"
)

discharge_summary

In [ ]:
# Candidate exclusion categories for investigation
death_codes = [11, 19, 20, 21]
hospice_codes = [13, 14]
uncertain_codes = [18, 25, 26]

disposition = work["discharge_disposition_id"]

print("Total encounters:", len(work))

print(
    "Death-related dispositions:",
    disposition.isin(death_codes).sum()
)

print(
    "Hospice dispositions:",
    disposition.isin(hospice_codes).sum()
)

print(
    "Unspecified or invalid dispositions:",
    disposition.isin(uncertain_codes).sum()
)

print(
    "Other discharge categories:",
    (~disposition.isin(
        death_codes + hospice_codes + uncertain_codes
    )).sum()
)

In [ ]:

# Group discharge disposition categories

death_codes = [11, 19, 20, 21]
hospice_codes = [13, 14]
uncertain_codes = [18, 25, 26]

work["discharge_group"] = "Other"

work.loc[
    work["discharge_disposition_id"].isin(death_codes),
    "discharge_group"
] = "Death-related"

work.loc[
    work["discharge_disposition_id"].isin(hospice_codes),
    "discharge_group"
] = "Hospice"

work.loc[
    work["discharge_disposition_id"].isin(uncertain_codes),
    "discharge_group"
] = "Unspecified/invalid"

# Compare original readmission outcomes
outcome_comparison = pd.crosstab(
    work["discharge_group"],
    work["readmitted"],
    margins=True
)

outcome_comparison


In [ ]:

# Calculate percentages within each discharge group

outcome_percentages = pd.crosstab(
    work["discharge_group"],
    work["readmitted"],
    normalize="index"
).mul(100).round(2)

outcome_percentages


In [ ]:

# Define discharge disposition groups

death_codes = [11, 19, 20, 21]
hospice_codes = [13, 14]
uncertain_codes = [18, 25, 26]

work["discharge_group"] = "Other"

work.loc[
    work["discharge_disposition_id"].isin(death_codes),
    "discharge_group"
] = "Death-related"

work.loc[
    work["discharge_disposition_id"].isin(hospice_codes),
    "discharge_group"
] = "Hospice"

work.loc[
    work["discharge_disposition_id"].isin(uncertain_codes),
    "discharge_group"
] = "Unspecified/invalid"

print(work["discharge_group"].value_counts())


In [ ]:

outcome_comparison = pd.crosstab(
    work["discharge_group"],
    work["readmitted"],
    margins=True
)

outcome_comparison


In [ ]:

outcome_percentages = pd.crosstab(
    work["discharge_group"],
    work["readmitted"],
    normalize="index"
).mul(100).round(2)

outcome_percentages


In [ ]:
outcome_comparison = pd.crosstab(
    work["discharge_group"],
    work["readmitted"],
    margins=True
)

outcome_comparison

In [ ]:
outcome_percentages = pd.crosstab(
    work["discharge_group"],
    work["readmitted"],
    normalize="index"
).mul(100).round(2)

outcome_percentages

In [ ]:
!git status

In [ ]:
!git add notebooks/python/02_cohort_and_data_dictionary.ipynb

In [ ]:
!cd

In [ ]:
!git add 02_cohort_and_data_dictionary.ipynb

In [ ]:
!git status

In [ ]:
!git commit -m "analysis: define and construct TRIP study cohort"

In [ ]:
!git push

In [ ]:
outcome_comparison = pd.crosstab(
    work["discharge_group"],
    work["readmitted"],
    margins=True
)

outcome_comparison

In [ ]:
outcome_percentages = pd.crosstab(
        work["discharge_group"],
        work["readmitted"],
        normalize="index"
    
).mul(100).round(2)

outcome_percentages



In [ ]:
outcome_percentages

In [ ]:
discharge_summary = (
    work.groupby("discharge_group")
    .agg(
        encounters=("encounter_id", "count"),
        unique_patients=("patient_nbr", "nunique"),
        readmissions_30d=("readmission_30d", "sum")
    )
    .reset_index()
)

discharge_summary